# 02 - Diversity indices

One metric, one method. **Simpson's standardized Dz** (0..1), computed **per outlet-week**, then
**averaged across outlets** for the country-week score (equal weight per outlet). Each grain is
also produced **split by topic** (immigration / inflation) for RQ3.

- *Societal* Dz: four actor categories. Open-diversity score, 1 = equal four-way.
- *Political* Dz: government vs opposition (1 = 50/50), carried with `p_gov` and the reflective
  benchmark (`dev_reflective` = p_gov - government seat share) for H2.

All datasets are written to `results/` in the final Save section, ready for modelling in `04`.

In [ ]:
import pandas as pd

import helpers as h

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

soc = pd.read_csv(h.SOCIAL_CLEAN, sep=h.CSV_SEP)
pol = pd.read_csv(h.POLITICAL_CLEAN, sep=h.CSV_SEP)
soc["topic_label"] = soc["topic"].map(h.TOPIC_LABEL).fillna(soc["topic"])
pol["topic_label"] = pol["topic"].map(h.TOPIC_LABEL).fillna(pol["topic"])

# one campaign_period label per country-week (a single boundary week carries both)
camp = (pd.concat([soc[["country", "year_week", "campaign_period"]],
                   pol[["country", "year_week", "campaign_period"]]])
          .groupby(["country", "year_week"])["campaign_period"].max().reset_index())

def add_camp(df):
    return df.merge(camp, on=["country", "year_week"], how="left")

refl = h.reflective_gov_share(pol).reset_index()   # country, reflective_gov_share
print(f"societal articles: {len(soc):,} | political actor-rows: {len(pol):,}")

## 1. Outlet-week Dz (overall and by topic)

The building block: one aggregated Dz per outlet per week, plain and split by topic.

In [ ]:
OUTLET     = ["country", "brand", "type", "year_week"]
OUTLET_TOP = ["country", "brand", "type", "topic_label", "year_week"]

def political(keys):
    out = add_camp(h.political_dz(pol, keys)).merge(refl, on="country", how="left")
    out["dev_reflective"] = out["p_gov"] - out["reflective_gov_share"]
    return out

soc_ow  = add_camp(h.societal_dz(soc, OUTLET))
soc_owt = add_camp(h.societal_dz(soc, OUTLET_TOP))
pol_ow  = political(OUTLET)
pol_owt = political(OUTLET_TOP)
print("outlet-week rows  | societal:", len(soc_ow), "/", len(soc_owt),
      " political:", len(pol_ow), "/", len(pol_owt))
pol_ow.head()

## 2. Country-week Dz, averaged across outlets (overall and by topic)

Each country-week (or country-week-topic) is the mean of its outlet-week scores. `n_outlets`
flags weeks with fewer than four contributing outlets; `n_actors` is the total actor volume.

In [ ]:
def to_country(ow, value_cols, by_topic=False):
    keys = ["country"] + (["topic_label"] if by_topic else []) + ["year_week"]
    g = ow.groupby(keys)
    out = g[value_cols].mean().reset_index()
    out["n_outlets"] = g["simpson_Dz"].count().values
    out["n_actors"]  = g["n_actors"].sum().values
    return add_camp(out)

SOC_VALS = ["simpson_Dz"]
POL_VALS = ["simpson_Dz", "p_gov", "dev_reflective"]

soc_cw  = to_country(soc_ow,  SOC_VALS)
soc_cwt = to_country(soc_owt, SOC_VALS, by_topic=True)
pol_cw  = to_country(pol_ow,  POL_VALS)
pol_cwt = to_country(pol_owt, POL_VALS, by_topic=True)
print("country-week rows | societal:", len(soc_cw), "/", len(soc_cwt),
      " political:", len(pol_cw), "/", len(pol_cwt))
pol_cw.head()

## 2b. Country-week Dz, aggregated (for comparison)

The alternative country score: pool every actor across all outlets in the country-week into one
distribution and index once. This is **volume-weighted**, so high-volume outlets (e.g. Cumhuriyet
in TR inflation) dominate. Kept for comparison against the outlet-averaged scores above; the
outlet-averaged version remains primary.

In [ ]:
soc_cw_pool  = add_camp(h.societal_dz(soc, ["country", "year_week"]))
soc_cwt_pool = add_camp(h.societal_dz(soc, ["country", "topic_label", "year_week"]))
pol_cw_pool  = political(["country", "year_week"])
pol_cwt_pool = political(["country", "topic_label", "year_week"])
print("country-aggregated rows | societal:", len(soc_cw_pool), "/", len(soc_cwt_pool),
      " political:", len(pol_cw_pool), "/", len(pol_cwt_pool))

## 2c. PSB vs newspaper paired scores (country-week)

For RQ2: per country-week, the public broadcaster's diversity and the equal-weighted mean of the newspapers' diversity, paired within the same country-week. This puts RQ2 at the country-week grain and supports the within-system PSB-vs-newspaper comparison.

In [ ]:
# Per country-week, two rows: the public broadcaster and the newspapers (equal-weighted
# mean), as a "group" factor, in the same long format (simpson_Dz) as the other datasets.
def psb_news_paired(ow):
    group = ow["type"].map(lambda t: "Public broadcaster" if t == "Public broadcaster" else "Newspaper")
    out = (ow.assign(group=group)
             .groupby(["country", "year_week", "group"])["simpson_Dz"].mean().reset_index())
    return add_camp(out)

soc_psb_news = psb_news_paired(soc_ow)
pol_psb_news = psb_news_paired(pol_ow)
soc_psb_news.to_csv(h.RESULTS_DIR / "soc_div_psb_news_paired.csv", index=False)
pol_psb_news.to_csv(h.RESULTS_DIR / "pol_div_psb_news_paired.csv", index=False)
print("saved *_div_psb_news_paired.csv:", len(soc_psb_news), "/", len(pol_psb_news), "rows")
soc_psb_news.head()

## 3. Save datasets for modelling

All diversity datasets written to `results/`. The `_aggregated` files are the country-aggregated
comparison; the plain country-week files are the outlet-averaged primary. Inputs for `04`.

In [ ]:
datasets = {
    "soc_div_outlet_week":               soc_ow,
    "soc_div_outlet_week_topic":         soc_owt,
    "soc_div_country_week":              soc_cw,
    "soc_div_country_week_topic":        soc_cwt,
    "soc_div_country_week_aggregated":       soc_cw_pool,
    "soc_div_country_week_topic_aggregated": soc_cwt_pool,
    "pol_div_outlet_week":               pol_ow,
    "pol_div_outlet_week_topic":         pol_owt,
    "pol_div_country_week":              pol_cw,
    "pol_div_country_week_topic":        pol_cwt,
    "pol_div_country_week_aggregated":       pol_cw_pool,
    "pol_div_country_week_topic_aggregated": pol_cwt_pool,
    "pol_reflective_gov_share":          refl,
}

h.RESULTS_DIR.mkdir(exist_ok=True)
for name, df in datasets.items():
    df.to_csv(h.RESULTS_DIR / f"{name}.csv", index=False)
    print(f"saved {name:<28} {len(df):>5} rows")

## 4. Preview: outlet-averaged vs country-aggregated

Mean weekly Dz per country, both country methods side by side, against political-independence
risk. Where they diverge most (Turkey) is where one outlet's volume dominates the aggregated score.

In [ ]:
risk = soc.groupby("country")["Pol_Independence"].first()
def cmean(df, col): return df.groupby("country")[col].mean().round(3)
prev = pd.DataFrame({
    "risk": risk,
    "soc_Dz_avg":    cmean(soc_cw, "simpson_Dz"),
    "soc_Dz_aggregated": cmean(soc_cw_pool, "simpson_Dz"),
    "pol_Dz_avg":    cmean(pol_cw, "simpson_Dz"),
    "pol_Dz_aggregated": cmean(pol_cw_pool, "simpson_Dz"),
}).reindex(h.COUNTRY_ORDER)
print(prev.to_string())